# Strategic Funnel: Marketing, Conversion & Returns
**Datathon 2026 - Growth & Profitability Analysis**

This notebook analyzes the enterprise funnel from traffic acquisition to revenue leakage. We combine `web_traffic.csv`, `promotions.csv`, `orders.csv`, and `returns.csv`.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Project configuration
PROCESSED_DIR = Path("../data/processed")
PLOT_DIR = Path("../data/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
pd.options.display.max_columns = None

## 1. Data Loading
Merging traffic, promos, and returns.

In [2]:
print("Loading datasets...")
traffic = pd.read_parquet(PROCESSED_DIR / "web_traffic.parquet")
promos = pd.read_parquet(PROCESSED_DIR / "promotions.parquet")
orders = pd.read_parquet(PROCESSED_DIR / "orders.parquet")
returns = pd.read_parquet(PROCESSED_DIR / "returns.parquet")
order_items = pd.read_parquet(PROCESSED_DIR / "order_items.parquet")

# Date handling
traffic['date'] = pd.to_datetime(traffic['date'])
traffic['year_month'] = traffic['date'].dt.to_period('M').astype(str)
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['year_month'] = orders['order_date'].dt.to_period('M').astype(str)
returns['return_date'] = pd.to_datetime(returns['return_date'])

print("Calculating monthly aggregates...")
monthly_traffic = traffic.groupby('year_month').agg(
    total_sessions=('sessions', 'sum'),
    avg_bounce_rate=('bounce_rate', 'mean')
).reset_index()

monthly_orders = orders.groupby('year_month').agg(
    order_count=('order_id', 'count')
).reset_index()

funnel_df = monthly_traffic.merge(monthly_orders, on='year_month', how='left')
funnel_df['conversion_rate'] = (funnel_df['order_count'] / funnel_df['total_sessions']) * 100

print("Funnel base ready.")

Loading datasets...
Calculating monthly aggregates...
Funnel base ready.


## 2. Traffic vs Conversion Trend
### Diagnostic Analysis
Did the traffic drop, or did customers stop buying?

In [3]:
fig = go.Figure()
fig.add_trace(go.Bar(x=funnel_df['year_month'], y=funnel_df['total_sessions'],
                     name='Monthly Sessions', marker_color='lightblue'))

fig.add_trace(go.Scatter(x=funnel_df['year_month'], y=funnel_df['conversion_rate'],
                         mode='lines+markers', name='Conversion Rate (%)',
                         yaxis='y2', line=dict(color='orange', width=3)))

fig.update_layout(
    title='Marketing Efficiency: Sessions vs Conversion Rate',
    yaxis=dict(title='Total Sessions'),
    yaxis2=dict(title='Conversion Rate (%)', overlaying='y', side='right', range=[0, 10]),
    legend=dict(x=0.01, y=0.99)
)

fig.write_image(PLOT_DIR / "12_traffic_conversion.png")
fig.show()

**Insight:** 
A drop in `Total Sessions` implies a marketing failure. A drop in `Conversion Rate` with stable sessions implies a website UX issue (high bounce rate) or unappealing products/prices.

## 3. Promotion ROI & Impact
### Diagnostic Analysis
Which promo types drive the most conversion?

In [4]:
promo_stats = promos.groupby('promo_type').agg(
    avg_discount=('discount_value', 'mean'),
    count=('promo_id', 'count')
).reset_index().sort_values('avg_discount', ascending=False)

fig = px.bar(promo_stats, x='promo_type', y='avg_discount', 
             color='count', title='Average Discount by Promo Type')
fig.write_image(PLOT_DIR / "13_promo_impact.png")
fig.show()

## 4. Revenue Leakage: Returns Analysis
### Diagnostic Analysis
Why are we losing money?

In [5]:
return_reasons = returns['return_reason'].value_counts().reset_index()
return_reasons.columns = ['reason', 'count']

fig = px.pie(return_reasons, values='count', names='reason', 
             title='Primary Reasons for Returns',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.write_image(PLOT_DIR / "14_return_reasons.png")
fig.show()

**Prescriptive:** 
If 'wrong_size' is the leader, we need an AI-driven size recommender. If 'defective' is high, supplier quality audits are mandatory.